In [1]:
import pandas as pd
import cassiopeia as cas
from tqdm import tqdm
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

In [2]:
metadata = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/cellranger/scRNA.metadata.csv',header=0,index_col=0)
metadata.index = metadata.cellName

/tmp/ipykernel_99303/2476779101.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/cellranger/scRNA.metadata.csv',header=0,index_col=0)


In [3]:
metadata

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,RNA_snn_res.0.2,seurat_clusters,pANN_0.25_0.12_592,Doublet_or_Singlet,pANN_0.25_0.21_600,pANN_0.25_0.19_742,...,pANN_0.25_0.07_798,RNA_snn_res.0.3,harmony_clusters,seurat_clusters_rename,umapharmony_1,umapharmony_2,cellName,lineageGrp,tumor_state,color
cellName,,,,,,,,,,,,,,,,,,,,,
T1_a3026_AAACCCAAGACTAGAT,T1_a3026,23101,4348,5.653435,1.0,1,0.314012,Singlet,NaN,NaN,...,NaN,1,1,2,3.960447,5.419786,T1_a3026_AAACCCAAGACTAGAT,NaN,2,NaN
T1_a3026_AAACCCACATGAGATA,T1_a3026,13159,3546,2.029030,0.0,2,0.356351,Singlet,NaN,NaN,...,NaN,0,2,1,-4.321606,-2.761532,T1_a3026_AAACCCACATGAGATA,C105,1,NaN
T1_a3026_AAACCCAGTCGCTGCA,T1_a3026,15441,3362,5.984068,0.0,2,0.256048,Singlet,NaN,NaN,...,NaN,0,2,1,-2.283126,-2.055104,T1_a3026_AAACCCAGTCGCTGCA,NaN,1,NaN
T1_a3026_AAACCCAGTCTCGCGA,T1_a3026,87500,7440,4.874286,0.0,2,0.452117,Singlet,NaN,NaN,...,NaN,0,2,1,-2.347915,-1.723554,T1_a3026_AAACCCAGTCTCGCGA,NaN,1,NaN
T1_a3026_AAACCCATCACGGGAA,T1_a3026,22915,3939,4.233035,0.0,2,0.250000,Singlet,NaN,NaN,...,NaN,0,2,1,-2.880527,-2.335177,T1_a3026_AAACCCATCACGGGAA,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T3_a3030_1_1_TTTGTTGCATGGACAG,T3_a3030_1_1,21144,4395,7.609724,NaN,0,NaN,Singlet,NaN,NaN,...,0.269478,3,0,3,-1.273801,-2.477038,T3_a3030_1_1_TTTGTTGCATGGACAG,C5,3,NaN
T3_a3030_2_1_TTTGTTGGTAAGATCA,T3_a3030_2_1,15115,4228,5.339067,NaN,0,NaN,Singlet,NaN,NaN,...,0.369386,2,0,3,0.885944,-1.561220,T3_a3030_2_1_TTTGTTGGTAAGATCA,NaN,3,NaN
T3_a3030_1_1_TTTGTTGGTGCGGTAA,T3_a3030_1_1,12252,2836,5.811296,NaN,2,NaN,Singlet,NaN,NaN,...,0.109991,4,2,1,-5.376790,-0.871353,T3_a3030_1_1_TTTGTTGGTGCGGTAA,NaN,1,NaN


In [4]:
clone_tree_list = pd.read_csv("clone.newick.list",sep='\t',header=None)
for clone_tree in tqdm(clone_tree_list.iloc[:,0].values):  
    cas_tree = cas.data.CassiopeiaTree(tree=clone_tree,root_sample_name='synthetic')
    cas_tree.remove_leaves_and_prune_lineages(set(cas_tree.leaves).difference(set(metadata.index.to_list())))
    cas_tree.cell_meta = pd.DataFrame(metadata.loc[[leaf for leaf in cas_tree.leaves if leaf !='synthetic'],"tumor_state"].astype(str))
    cas_tree.cell_meta.loc['synthetic'] = '-1'
    
    parsimony = cas.tl.score_small_parsimony(cas_tree, meta_item="tumor_state")
    plasticity = parsimony / len(cas_tree.nodes)
    # compute plasticities for each node in the tree
    for node in cas_tree.depth_first_traverse_nodes():
        effective_plasticity = cas.tl.score_small_parsimony(
            cas_tree, meta_item="tumor_state", root=node
        )
        size_of_subtree = len(cas_tree.leaves_in_subtree(node))
        cas_tree.set_attribute(
            node, "effective_plasticity", effective_plasticity / size_of_subtree
        )

    cas_tree.cell_meta["scPlasticity"] = 0
    for leaf in cas_tree.leaves:
        plasticities = []
        parent = cas_tree.parent(leaf)
        while True:
            plasticities.append(cas_tree.get_attribute(parent, "effective_plasticity"))
            if parent == cas_tree.root:
                break
            parent = cas_tree.parent(parent)
        cas_tree.cell_meta.loc[leaf, "scPlasticity"] = np.mean(plasticities)        
    cas_tree.cell_meta.to_csv(clone_tree+'.plasticity')


100%|██████████| 31/31 [00:01<00:00, 24.83it/s]
